In [0]:
#1 Definamos las variables de nuestro entorno (unity Catalog)
catalog_name = "bronze_dev"
schaema_name = "rrhh"
table_name = "clients_brz"

#concatenamossiguiendo el estardad de 3 niveles
full_table_name = f"{catalog_name}.{schaema_name}.{table_name}"


#ruta tecnica para el manejo de estado striming
checkpoint_path = f"s3://lakehouse-databricks-dev/bronze-dev/_pipelines/{table_name}/_checkpoint/"
schema_path = f"s3://lakehouse-databricks-dev/bronze-dev/_pipelines/{table_name}/_schema/"
landing_path = 's3://lakehouse-databricks-dev/landing-dev/clients/'



In [0]:
#df=spark.read.option("header","true").csv(landing_path)

In [0]:
df_bronze_raw= (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header","true")
    .load(landing_path)

)

In [0]:
query_production = (
    df_bronze_raw.
    writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow =True)
    .table(full_table_name)

)

In [0]:
df = spark.read.table(full_table_name)
display(df)